# Application: turn browser receipts into Python values

Run this unrun tutorial cell by cell in nbinlineai JupyterLab with a live Python kernel. Camera and microphone calls may show browser permission prompts; choose a safe scene and say a short test phrase. Each browser call returns immediately. Move to its later inspection cell after the visible operation finishes, and rerun that inspection if still active. No cell waits inside the kernel. AI questions use a configured model and may incur charges. There are no saved outputs or AI answers in this source notebook.

A regular Python function returns its value at once. A browser receipt is mutable: `status` and `result` can change after the call cell has finished. The printed result of the call cell is just an early snapshot.

These media APIs require a Git-source installation containing them; PyPI 0.1.15 does not include them. Use your notebook’s selected image-capable connection for image questions.


In [ ]:
from pathlib import Path
from uuid import uuid4
from PIL import Image
from nbinlineai.browser_receipt import MediaClip
from nbinlineai.tools import (browser_capabilities, operation_status, start_camera,
    capture_camera, stop_source, record_microphone, release_media, attach_media)

work = Path.cwd() / f'receipts-demo-{uuid4().hex[:10]}'
work.mkdir()  # An owned Python output folder; browser captures stay in memory.

def require_complete(receipt, label):
    print(label, 'state:', receipt.status, 'operation:', receipt.operation_id)
    if receipt.status != 'completed':
        raise RuntimeError(f'{label}: inspect again after the browser finishes; error={receipt.error}')
    return receipt.result

def with_tax(price: float, rate: float) -> float:
    """Return a tax-inclusive price as an ordinary Python number."""
    return round(price * (1 + rate), 2)

ordinary_value = with_tax(12.50, 0.08)
print('Ordinary Python value:', ordinary_value)


Available to the next AI question:
- &`with_tax` — return a Python number from an ordinary function.

This ordinary Markdown cell is a tool declaration. Its Tools control should be on for the next question. The function must still exist in the live kernel.


Call `with_tax` once with `price=12.50` and `rate=0.08`. Report the returned number and say that it is an ordinary Python value, with no browser operation ID.


For the browser questions below, turn **Tools off** on the earlier `with_tax` declaration if you want a smaller menu. The next questions declare only `browser_capabilities` and `operation_status` directly; the unique inherited tool count stays well below 20.


In [ ]:
capability_receipt = browser_capabilities()
capability_receipt  # Early rendered state; this output will not refresh itself.


In [ ]:
capability_data = require_complete(capability_receipt, 'capabilities')
print('Secure browser context:', capability_data['secure_context'])
print('Memory recording limit (seconds):', capability_data['limits']['recording_memory_seconds'])


In [ ]:
status_receipt = operation_status(capability_receipt.operation_id)
status_receipt


In [ ]:
snapshot = require_complete(status_receipt, 'status lookup')
print('Lookup receipt:', status_receipt.status, status_receipt.operation_id)
print('Target snapshot:', snapshot['status'], snapshot['operation_id'])
assert snapshot['operation_id'] == capability_receipt.operation_id


`operation_status` has **two states**: its own lookup receipt is `completed`, while `status_receipt.result` is a snapshot of the target. A completed lookup can report a still-running target. This snapshot contains metadata, not decoded media bytes. Inspect the original capture receipt later for `.result`.


Call &`browser_capabilities` once. Give its actual operation ID and current state. If the call has not completed, stop there; do not claim a limit yet.


After the prior browser capability operation finishes, call &`operation_status` using that **exact operation ID** from the previous answer. Report both the lookup state and the target snapshot state, and one actual memory recording limit only if the target completed. This AI call does not assign Python variables.


## Camera image: descriptor and Pillow value

Grant camera access only if you choose. Start a source, inspect it on a later kernel turn, then capture a still. `save_to=None` keeps the browser result in memory; no server-relative file is generated.


In [ ]:
camera_start = start_camera(audio=False)
camera_start


In [ ]:
camera_info = require_complete(camera_start, 'camera start')
camera_source_id = camera_info['source_id']
print('Active camera source:', camera_source_id)


In [ ]:
photo_receipt = capture_camera(source_id=camera_source_id, save_to=None, max_size=960)
photo_receipt


In [ ]:
photo = require_complete(photo_receipt, 'camera photo')
assert isinstance(photo, Image.Image)
assert isinstance(photo_receipt.media, dict)
photo_ref = {'media_id': photo_receipt.media['media_id']}
print('Descriptor keys:', sorted(photo_receipt.media))
print('Real Pillow image:', photo.size, photo.mode)
photo_path = work / 'camera-photo.png'
photo.save(photo_path, format='PNG')
print('Python copy:', photo_path)
display(photo)


`photo_receipt.media` is a managed descriptor with an ID, MIME type, hash and byte count. It can identify media for another browser tool, but is not the image itself. `photo_receipt.result` is the decoded Pillow image. The Python copy above survives a later `release_media` call. `operation_status(photo_receipt.operation_id)` would return a snapshot with the descriptor, not this Pillow object.


In [ ]:
camera_stop = stop_source(camera_source_id)
camera_stop


In [ ]:
print('Stop control result:', require_complete(camera_stop, 'camera stop'))


## Microphone clip: descriptor and encoded bytes

Speak a short phrase after granting microphone access. The bounded one-shot recording closes its own source. It does not transcribe speech. Use a later cell after its visible media row finishes.


In [ ]:
audio_receipt = record_microphone(save_to=None, duration=6)
audio_receipt


In [ ]:
clip = require_complete(audio_receipt, 'microphone recording')
assert isinstance(clip, MediaClip)
assert isinstance(audio_receipt.media, dict)
print('Clip MIME:', clip.mime_type, 'bytes:', len(clip.data), 'duration:', clip.duration_seconds)
print('Managed media ID:', audio_receipt.media['media_id'])
suffix = {'audio/webm': '.webm', 'audio/mp4': '.m4a', 'audio/ogg': '.ogg'}.get(clip.mime_type.split(';', 1)[0])
if suffix is None:
    raise ValueError(f'Unsupported audio MIME for a portable copy: {clip.mime_type}')
audio_path = work / f'microphone{suffix}'
audio_path.write_bytes(clip.data)
print('Python copy:', audio_path)


## Optional model view of the still

A tool result descriptor does not show the model photo pixels or audio. The direct call below asks for visible confirmation to attach **only this still** to the stable target question. Inspect the attachment receipt after you click Attach image; then run the target question yourself. Never attach a private capture without reviewing it first. Audio and video are not supported model attachments here.


In [ ]:
photo_attachment = attach_media(photo_ref, 'receipts-image-question', detail='auto')
photo_attachment


In [ ]:
print('Attachment confirmation:', require_complete(photo_attachment, 'photo attachment'))


Describe the attached camera still in one sentence. If no image indicator is present, say that you cannot inspect the pixels. Do not infer image content from the earlier `photo_receipt.media` descriptor.


## Release temporary browser media

Run after the image question and after the Python copies exist. These calls release only the two managed media IDs created here; they do not delete the files in `work`. Remove the optional image attachment with the question's UI before releasing it if you intend to rerun that question.


In [ ]:
photo_release = release_media(photo_receipt.media['media_id'])
audio_release = release_media(audio_receipt.media['media_id'])


In [ ]:
print('Photo release:', require_complete(photo_release, 'photo release'))
print('Audio release:', require_complete(audio_release, 'audio release'))
print('Kept Python files:', [p.name for p in work.iterdir()])
